# 3D bar chart run 1 — any 3D bar chart (Qwen3.5-9B + LoRA)

**Goal:** read every bar of a 3D bar chart — category (front axis), series (depth axis or legend) and value (height).

**Training** uses only synthetic 3D bar charts: 2–6 categories × 1–4 series (at most 16 bars), values spread out so no
two bars look alike, many topics, value ranges from 1 to 100,000, many colours, backgrounds, camera angles and walk-arounds,
sometimes a legend, **never numbers on the bars** (the model must read the heights). Answer: `Category,Series,Value`. All test names are banned (`--strict`).

**Test** (never used for training): 4 DxR bar charts with random values fixed before recording
(truth + sha256 in `ground_truth/control/MANIFEST_bar.json`):
the tested bar chart `bar3d_control_N`: test score, 2 classes × 4 rounds (decimals).

**Decided before testing:** the main result is the **median of 7-frame votes** (as for the line chart).

**Run one cell at a time; wait for ✓.** Cells 5–6 (data + training, ~3 h on T4) do not need the videos: record N
in DxR while it trains. Everything is saved in Drive (`MyDrive/dxr_bar`).

| Cell | What it does | T4 time |
|---|---|---|
| 1 | install, then the session restarts by itself | ~6 min |
| 2 | Drive + project zip (copied from `dxr_pie` if there) | 1 min |
| 3 | code `dxr_code_bar_r1c.tar.xz` (asked once) | seconds |
| 4 | settings | – |
| 5 | 1500 synthetic 3D bar charts + 3 shown | ~15 min |
| 6 | **train** — saves a checkpoint to Drive every 15 steps; if Colab stops, run cells 1–6 again and it resumes | ~4–5 h |
| 7 | test videos: upload N | 2 min |
| 8 | baseline: untrained model | ~30 min |
| 9 | test the trained model | ~1.5 h |
| 10 | results + proof videos + `bar_r1c_results.zip` | ~3 min |

In [ ]:
# 1 · Install (the session restarts by itself at the end: then run cell 2)
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \\
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \\
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \\
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install -q --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install -q transformers==5.2.0
!unsloth install-kernels
!uv pip install -q --no-deps --upgrade "torchao>=0.16.0"
!pip install -q -U bitsandbytes hf_transfer "json5>=0.9.25" "structlog>=24.1.0" tyro cut_cross_entropy msgspec
!pip check | grep -i unsloth || echo "unsloth deps OK"
!python3 -c "import unsloth, transformers; print('INSTALL OK - unsloth ready, transformers', transformers.__version__)"
print("Restarting the session now (needed after installing). Then run cell 2.")
import time; time.sleep(2); os.kill(os.getpid(), 9)

In [ ]:
# 2 · Google Drive + project zip (made on the Mac: zip -qr ~/Desktop/dxr_pie_upload.zip vlm_eval ground_truth)
from google.colab import files, drive
import zipfile, os, glob, shutil, json, csv
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/dxr_bar"
os.makedirs(DRIVE, exist_ok=True)
for z in ["/content/drive/MyDrive/dxr_pie/dxr_pie_upload.zip"]:
    if os.path.exists(z) and not os.path.exists(f"{DRIVE}/dxr_pie_upload.zip"): shutil.copy(z, DRIVE)
if not os.path.exists("/content/dxr/vlm_eval/prompts.md"):
    src = f"{DRIVE}/dxr_pie_upload.zip"
    if not os.path.exists(src):
        print("Choose dxr_pie_upload.zip (or dxr_video_upload.zip) from your Desktop")
        up = files.upload(); shutil.copy(next(iter(up)), src)
    zipfile.ZipFile(src).extractall("/content/dxr")
os.chdir("/content/dxr")
for f in ["results_bar_r1c.csv"]:
    if os.path.exists(f"{DRIVE}/{f}") and not os.path.exists(f): shutil.copy(f"{DRIVE}/{f}", f)
if os.path.exists(f"{DRIVE}/predictions") and not os.path.exists("predictions"): shutil.copytree(f"{DRIVE}/predictions", "predictions")
!ls; ls ground_truth
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3 · Code for bar run 1. Asked once: choose dxr_code_bar_r1c.tar.xz from Downloads; then kept in Drive
%cd /content/dxr
import tarfile
code_src = f"{DRIVE}/dxr_code_bar_r1c.tar.xz"
if not os.path.exists(code_src):
    print("Choose dxr_code_bar_r1c.tar.xz")
    up = files.upload(); shutil.copy(next(iter(up)), code_src)
tarfile.open(code_src, "r:xz").extractall("/content/dxr")
ok = "--save-dir" in open("finetune_video/train_video_lora.py").read() and os.path.exists("ground_truth/control/bar3d_control_N.csv")
print("code bar r1 OK" if ok else "WRONG CODE FILE - choose dxr_code_bar_r1c.tar.xz")
!cat ground_truth/control/bar3d_control_N.csv

In [ ]:
# 4 · Settings
%cd /content/dxr
import torch
CHART, MODEL = "bar3d", "unsloth/Qwen3.5-9B"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if gb > 30:
    TRAIN, STEPS, SIDE, MAXLEN, X, VALN = 2000, 250, 800, 12288, "", 10
else:
    TRAIN, STEPS, SIDE, MAXLEN, X, VALN = 1500, 120, 640, 6144, "--load-4bit", 5   # T4: ~2 min/step; loss levels off by ~60 steps
ADAPTER, RUN = f"{DRIVE}/lora_qwen35_bar_r1c", "qwen35_bar_r1c"
DATA, REAL, RES = f"finetune_video/data_bar3d{SIDE}", "real_clips_bar3d/real.jsonl", "results_bar_r1c.csv"
VDIR = f"{DRIVE}/videos"
def save():
    if os.path.exists(RES): shutil.copy(RES, DRIVE)
    if os.path.exists("predictions"): shutil.copytree("predictions", f"{DRIVE}/predictions", dirs_exist_ok=True)
def done(model, inp, video):
    return os.path.exists(RES) and any(r["model"] == model and r["set"] == "real" and r["input"] == inp and r["clip"] == video
                                       for r in csv.DictReader(open(RES)))
print(f"GPU {gb:.0f} GB | {TRAIN} clips, {STEPS} steps, {SIDE} px | {X or '16-bit'}")

In [ ]:
# 5 · Synthetic 3D bar charts (strict: test names banned) — ~15 min on T4, do not stop it
%cd /content/dxr
from PIL import Image
from IPython.display import display, Markdown
if not os.path.exists(f"{DATA}/val.jsonl") or sum(1 for _ in open(f"{DATA}/train.jsonl")) < TRAIN:
    shutil.rmtree(DATA, ignore_errors=True)
    !python finetune_video/make_video_dataset.py --strict --charts bar3d --train {TRAIN} --val 30 --size {SIDE}x{SIDE*3//4} --out {DATA}
def strip(paths, base):
    ims = [Image.open(f"{base}/{p}").resize((240, 180)) for p in paths]
    s = Image.new("RGB", (240 * len(ims), 180), "white")
    for i, im in enumerate(ims): s.paste(im, (240 * i, 0))
    return s
banned = json.load(open(f"{DATA}/excluded_names.json"))
print(sum(1 for _ in open(f"{DATA}/train.jsonl")), "training clips ready |", len(banned), "real/test names banned, e.g.",
      [n for n in ("Spring", "Library", "Monday", "Round 1") if n in banned])
for k, l in enumerate(open(f"{DATA}/train.jsonl").readlines()[:3]):
    r = json.loads(l); print(f"synthetic training clip {k + 1} (made-up data):", r["answer"].split(chr(10))[1:4], "...")
    display(strip(r["frames"], DATA))

In [ ]:
# 6 · Train on synthetic 3D bar charts only. Keep this tab open and the Mac awake (`caffeinate -i` in Terminal)
%cd /content/dxr
if not os.path.exists(f"{ADAPTER}/adapter_config.json"):
    shutil.rmtree("lora_qwen35_bar_r1c", ignore_errors=True)
    !python finetune_video/train_video_lora.py --base {MODEL} --data {DATA} --steps {STEPS} --max-side {SIDE} --max-length {MAXLEN} --out lora_qwen35_bar_r1c {X} --save-dir {DRIVE}/ckpt_bar_r1c
    shutil.copytree("lora_qwen35_bar_r1c", ADAPTER, dirs_exist_ok=True)
print(open(f"{ADAPTER}/training_info.json").read())

In [ ]:
# 7 · Test videos. Name your recording bar3d_control_N.mp4 before uploading. Re-run to add more later.
%cd /content/dxr
os.makedirs(VDIR, exist_ok=True)
need = [c for c in ("bar3d_control_N",) if not glob.glob(f"{VDIR}/{c}.*")]
if need:
    print("Choose the recordings:", need)
    up = files.upload()
    for n in up:
        if n.startswith("bar3d_control_"): shutil.copy(n, f"{VDIR}/{n}"); print("saved", n)
        else: print("SKIPPED", n, "- rename it to bar3d_control_N.mp4 on the Mac and upload again")
shutil.rmtree("real_clips_bar3d", ignore_errors=True)
!python3 finetune_video/sample_real_videos.py {VDIR} --charts bar3d --width {SIDE} --out real_clips_bar3d
VIDEOS = [v for v in (json.loads(l)["video"] for l in open(REAL)) if os.path.exists(f"ground_truth/control/{v}.csv")]
print("testing:", VIDEOS)
for l in open(REAL):
    r = json.loads(l); print("test video:", r["video"]); display(strip(r["frames"], "real_clips_bar3d"))

In [ ]:
# 8 · Baseline: the untrained Qwen3.5-9B on every test video
%cd /content/dxr
todo = [v for v in VIDEOS if not done("qwen35_base", "video", v)]
if todo:
    !python finetune_video/eval_video.py --model {MODEL} --name qwen35_base --charts bar3d --real {REAL} --videos {" ".join(todo)} --val {DATA}/val.jsonl --val-n 3 --max-side {SIDE} {X} --results {RES}
    save()
print("baseline done for", VIDEOS)

In [ ]:
# 9 · Test the trained model: whole video + median of 7-frame votes (main result = median)
%cd /content/dxr
todo = [v for v in VIDEOS if not done(RUN, "median", v)]
if todo:
    !python finetune_video/eval_median.py --model {ADAPTER} --name {RUN} --charts bar3d --real {REAL} --videos {" ".join(todo)} --val {DATA}/val.jsonl --val-n {VALN} --max-side {SIDE} {X} --results {RES}
    save()
print("tested:", VIDEOS)

In [ ]:
# 10 · Results vs the true tables + proof videos + download
%cd /content/dxr
import sys; sys.path.insert(0, "control_test"); import make_proof_video as mpv
names = {("qwen35_base", "video"): "Untrained Qwen3.5-9B", (RUN, "video"): "Fine-tuned, whole video",
         (RUN, "median"): "**Fine-tuned, median of 7-frame votes (main)**"}
pred_of = {("qwen35_base", "video"): "predictions/video_qwen35_base/real_video_{v}.csv",
           (RUN, "video"): f"predictions/median_{RUN}/real_video_{{v}}.csv",
           (RUN, "median"): f"predictions/median_{RUN}/real_median_{{v}}.csv"}
fmt = lambda v, f: "–" if v is None else f.format(v)
os.makedirs("proof", exist_ok=True)
out, summary = [], ["| Test chart | Bars | Untrained within 5 % | Fine-tuned (main) within 5 % | Mean error |", "|---|---|---|---|---|"]
for v in VIDEOS:
    truth = f"ground_truth/control/{v}.csv"
    lines = [f"### {v}", "", "| Model | Within 5 % | Within 2 % | Mean error (% of value) | Average miss (units) | Missing |", "|---|---|---|---|---|---|"]
    got = {}
    for key, label in names.items():
        p = pred_of[key].format(v=v)
        if not os.path.exists(p): continue
        val, cmp_ = mpv.compare_labeled(mpv.read(truth), mpv.read(p)); st = mpv.stats(cmp_)
        pps = [abs(r["pp"]) for r in cmp_ if r["pp"] is not None]
        got[key] = (st, sum(pps) / len(pps) if pps else None, cmp_, p)
        lines.append(f"| {label} | **{100 * st['w5'] / st['n']:.1f} %** ({st['w5']}/{st['n']}) | {st['w2']}/{st['n']} | "
                     f"{st['mre']:.2f} % | {fmt(got[key][1], '{:.2f}')} | {st['missing']} |" if pps else f"| {label} | – | – | – | – | {st['missing']} |")
    main = got.get((RUN, "median"))
    if main:
        lines += ["", "**Per bar (main):**", "", "| Bar (category / series) | True | Model | Error | Miss |", "|---|---|---|---|---|"]
        for r in main[2]:
            lines.append(f"| {r['series']} | {r['truth']:g} | {fmt(r['model'], '{:g}')} | {fmt(r['err'], '{:+.1f} %')} | {fmt(r['pp'], '{:+g}')} |")
        vid = next(iter(glob.glob(f"{VDIR}/{v}.*")))
        !python3 control_test/make_proof_video.py --kind pie --video "{vid}" --truth {truth} --pred {main[3]} --frames real_clips_bar3d/{v} --model "Qwen3.5-9B + LoRA bar run 1 (synthetic 3D bars only), median of 7-frame votes" --out proof/{v}_proof.mp4
    b = got.get(("qwen35_base", "video"))
    summary.append(f"| {v} | {main[0]['n'] if main else '–'} | {fmt(100 * b[0]['w5'] / b[0]['n'] if b else None, '{:.1f} %')} | "
                   f"{fmt(100 * main[0]['w5'] / main[0]['n'] if main else None, '{:.1f} %')} | {fmt(main[0]['mre'] if main else None, '{:.2f} %')} |")
    out.append("\n".join(lines))
text = "## Bar run 1: summary\n\n" + "\n".join(summary) + "\n\n" + "\n\n".join(out)
display(Markdown(text))
for v in VIDEOS:
    if os.path.exists(f"proof/{v}_proof_result.png"): display(Image.open(f"proof/{v}_proof_result.png").resize((960, 540)))
open("proof/RESULTS_bar_r1.md", "w").write(text + "\n")
shutil.copytree("proof", f"{DRIVE}/proof", dirs_exist_ok=True); save()
!zip -qr bar_r1c_results.zip proof {RES} predictions/median_{RUN} predictions/video_qwen35_base ground_truth/control/MANIFEST_bar.json
shutil.copy("bar_r1c_results.zip", DRIVE)
files.download("bar_r1c_results.zip")

**If something goes wrong:** disconnected → run cells 1–4, then continue: finished steps say "already done" / skip.
Training already saved in Drive is never repeated. Send `bar_r1c_results.zip` for the slides.